In [0]:
%pip install geopandas leafmap --quiet

In [0]:
from pyspark.sql import functions as F

# Read municipal population data with geometries
df_all = spark.table("geospatial.spain_population_analysis.padron_municipios_geo")

# Find the latest period and filter to it (table contains multiple years)
latest_period = df_all.select(F.max("periodo")).collect()[0][0]
print(f"Latest period: {latest_period}")

df = df_all.filter(F.col("periodo") == latest_period)

# Sample to verify structure
print(f"Total municipalities: {df.count():,}")
total_pop = df.select(F.sum('poblacion')).collect()[0][0]
if total_pop:
    print(f"Total population: {total_pop:,}")

display(df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import explode, size
from pyspark.sql.window import Window

# Convert to H3 resolution 9 using Databricks H3 functions
# Resolution 9 provides detailed granularity (~4.8M cells nationally)
# Area-weighted population distribution: each cell gets population proportional to cell count

print("Converting geometries to H3 cells at resolution 9...")

# Polyfill and calculate cell count in one go
df_with_h3 = df.selectExpr(
    "codigo_municipio",
    "poblacion",
    "st_aswkt(geometry) as geom_wkt"
).selectExpr(
    "codigo_municipio",
    "poblacion",
    "h3_polyfillash3string(geom_wkt, 9) as h3_cells"
).withColumn(
    "num_cells",
    size("h3_cells")
)

print("Exploding H3 cells and distributing population...")

# Explode and distribute population evenly across cells
df_h3_exploded = df_with_h3.select(
    "codigo_municipio",
    "poblacion",
    "num_cells",
    explode("h3_cells").alias("h3_cell")
).withColumn(
    "poblacion_cell",
    F.col("poblacion") / F.col("num_cells")
)

print("Aggregating population by H3 cell...")

# Group by H3 cell to aggregate population
# NOT materializing geometries at res-9 (~4.8M cells) - too expensive
# Will create geometries only at res-7 and res-6 for visualization
df_h3_res9 = df_h3_exploded.groupBy("h3_cell").agg(
    F.sum("poblacion_cell").alias("poblacion")
).withColumn(
    "h3_resolution",
    F.lit(9)
).select(
    "h3_cell",
    "h3_resolution",
    "poblacion"
)

print(f"\nTotal H3 cells (res-9): {df_h3_res9.count():,}")
print(f"Total population in H3: {df_h3_res9.select(F.sum('poblacion')).collect()[0][0]:,.0f}")

display(df_h3_res9.limit(10))

In [0]:
# Save the H3 resolution 9 results to Unity Catalog for future use
table_name = "geospatial.spain_population_analysis.h3_poblacion_res9"

df_h3_res9.write.mode("overwrite").saveAsTable(table_name)

print(f"Saved H3 res-9 data to {table_name}")
print(f"Total rows: {df_h3_res9.count():,}")

In [0]:
# Roll up res-9 cells to resolution 7 and 6 with geometries for visualization
# Only materializing geometries at coarser resolutions (res-9 would be too expensive)
# Res-7 (~90K cells) and Res-6 (~13K cells) for national overview

print("Rolling up res-9 to res-7 for visualization...")

df_h3_res7 = df_h3_res9.selectExpr(
    "h3_toparent(h3_cell, 7) as h3_cell",
    "poblacion"
).groupBy("h3_cell").agg(
    F.sum("poblacion").alias("poblacion")
).withColumn(
    "h3_resolution",
    F.lit(7)
).selectExpr(
    "h3_cell",
    "h3_resolution",
    "poblacion",
    "h3_boundaryaswkb(h3_cell) as geometry"
)

print(f"Total H3 cells (res-7): {df_h3_res7.count():,}")
print(f"Total population (res-7): {df_h3_res7.select(F.sum('poblacion')).collect()[0][0]:,.0f}")

print("\nRolling up res-9 to res-6 for coarser visualization...")

df_h3_res6 = df_h3_res9.selectExpr(
    "h3_toparent(h3_cell, 6) as h3_cell",
    "poblacion"
).groupBy("h3_cell").agg(
    F.sum("poblacion").alias("poblacion")
).withColumn(
    "h3_resolution",
    F.lit(6)
).selectExpr(
    "h3_cell",
    "h3_resolution",
    "poblacion",
    "h3_boundaryaswkb(h3_cell) as geometry"
)

print(f"Total H3 cells (res-6): {df_h3_res6.count():,}")
print(f"Total population (res-6): {df_h3_res6.select(F.sum('poblacion')).collect()[0][0]:,.0f}")

# Save both resolutions for visualization
table_name_res7 = "geospatial.spain_population_analysis.h3_poblacion_res7"
df_h3_res7.write.mode("overwrite").saveAsTable(table_name_res7)
print(f"\nSaved H3 res-7 data to {table_name_res7}")

table_name_res6 = "geospatial.spain_population_analysis.h3_poblacion_res6"
df_h3_res6.write.mode("overwrite").saveAsTable(table_name_res6)
print(f"Saved H3 res-6 data to {table_name_res6}")

display(df_h3_res7.limit(10))

In [0]:
import geopandas as gpd
from shapely import wkb
import pandas as pd

# Convert H3 to GeoDataFrame for leafmap visualization
# Using res-7 for performance (res-9 with ~4.8M cells may be too heavy for interactive maps)
# Change to df_h3_res9 if you want the finest detail and have sufficient compute

pdf_h3 = df_h3_res7.toPandas()

# Convert WKB geometry to shapely geometries
pdf_h3['geometry'] = pdf_h3['geometry'].apply(lambda x: wkb.loads(bytes(x)))

# Create GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(pdf_h3, geometry='geometry', crs='EPSG:4326')

# Calculate density (population per km²)
gdf_h3['area_km2'] = gdf_h3.geometry.to_crs('EPSG:3035').area / 1_000_000
gdf_h3['densidad'] = gdf_h3['poblacion'] / gdf_h3['area_km2']

print(f"GeoDataFrame shape: {gdf_h3.shape}")
print(f"\nPopulation statistics:")
print(gdf_h3['poblacion'].describe())
print(f"\nDensity statistics (per km²):")
print(gdf_h3['densidad'].describe())

gdf_h3.head()

In [0]:
import leafmap
import matplotlib.pyplot as plt
import numpy as np

# Create map centered on Spain
m = leafmap.Map(center=[40.4, -3.7], zoom=6, height="800px")

# Filter out cells with very low population for cleaner visualization
gdf_h3_filtered = gdf_h3[gdf_h3['poblacion'] > 10].copy()

# Create classification bins for density (using quantiles for better distribution)
bins = gdf_h3_filtered['densidad'].quantile([0, 0.2, 0.4, 0.6, 0.8, 0.95, 1.0]).tolist()

# Add H3 choropleth layer
m.add_data(
    gdf_h3_filtered,
    column='densidad',
    scheme='UserDefined',
    classification_kwds={'bins': bins},
    cmap='YlOrRd',
    legend_title='Population Density (per km²)',
    layer_name='H3 Population Density (Res-7)',
    style_kwds={'fillOpacity': 0.7, 'weight': 0.1}
)

# Display the map
m